In [1]:
#!/usr/bin/env python3
"""AURORA ER timing audit for ALL patients with a metastatic ER result.

For every patient with (a) an informative metastatic ER assessment and (b) at least one
cfDNA draw that has Griffin data, this script calculates the time lag between the draw
and the first informative metastatic ER assessment, and plots how many patients fall
inside +/-30, 60, 90, 180 and 365 days.

Two draw rules are summarised side by side:
  * FIRST draw   : the earliest Griffin draw of the patient
  * CLOSEST draw : the Griffin draw nearest to the ER assessment (planned primary rule)

Lag convention: lag = draw day - ER assessment day (days from primary diagnosis).
Negative lag = blood drawn BEFORE the ER assessment.
Windows are cumulative (the +/-60 bar includes everyone inside +/-30).
"""
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# =============================================================================
# USER CONFIGURATION (edit paths here)
# =============================================================================
AURORA_BASE = Path("/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA")
CLINICAL_XLSX = AURORA_BASE / "Clinical_data" / "AUP_ClinicalData_20260609.xlsx"
# Metasheet is used ONLY to know which blood draws have Griffin/ULP-WGS data
META_XLSX = AURORA_BASE / "Clinical_data" / (
    "Metasheet_Analysis.with_Histology_Receptor_status.Earliest_Met_Diagnosis.xlsx")
OUT_DIR = (AURORA_BASE / "Griffin" / "Downstream" / "ER_phenotype"
           / "ER_timing_audit_all_patients")
OUT_DIR.mkdir(parents=True, exist_ok=True)

WINDOWS = [30, 60, 90, 180, 365]
COL_POS, COL_NEG = "#2C7FB8", "#F28E2B"          # ER+ (blue), ER- (orange)
OUT_XLSX = OUT_DIR / "AURORA_ER_timing_audit_all_patients.xlsx"


# =============================================================================
# Helper functions
# =============================================================================
def normalize_patient_id(value):
    """Standardize IDs that Excel may have read as numbers (2.0 -> '2')."""
    if pd.isna(value):
        return np.nan
    s = str(value).strip()
    try:
        f = float(s)
        if f.is_integer():
            return str(int(f))
    except ValueError:
        pass
    return s


def normalize_er(value):
    """Map ER text to Positive / Negative; anything else (Unknown, blank) -> missing."""
    if pd.isna(value):
        return np.nan
    v = str(value).strip().lower()
    if v in {"positive", "pos", "+", "er+"}:
        return "Positive"
    if v in {"negative", "neg", "-", "er-"}:
        return "Negative"
    return np.nan


def normalize_sample(value):
    """AUP-AMCZ-NB1 / AMCZ-NB1 -> AMCZ-NB1 so clinical and metasheet IDs match."""
    if pd.isna(value):
        return np.nan
    s = str(value).strip()
    return s[4:] if s.upper().startswith("AUP-") else s


def find_clinical_sheet(path):
    required = {"patient_id", "Category", "Days_From_Primary_Diagnosis", "ER_Status", "Barcode"}
    with pd.ExcelFile(path) as xls:
        for name in xls.sheet_names:
            cols = set(pd.read_excel(path, sheet_name=name, nrows=0).columns)
            if required.issubset(cols):
                return name
    raise ValueError(f"No worksheet contains all required columns: {sorted(required)}")


def window_table(df, lag_col, label):
    """Cumulative counts of patients within +/-N days, split by metastatic ER."""
    rows = []
    a = df[lag_col].abs()
    for w in WINDOWS:
        s = df[a <= w]
        rows.append(dict(Draw_rule=label, Window=f"±{w} days", Window_days=w,
                         Metastatic_ER_pos=int((s["Metastatic_ER"] == "Positive").sum()),
                         Metastatic_ER_neg=int((s["Metastatic_ER"] == "Negative").sum()),
                         Total=len(s)))
    s = df[a > WINDOWS[-1]]
    rows.append(dict(Draw_rule=label, Window=f"Outside ±{WINDOWS[-1]} days", Window_days=np.inf,
                     Metastatic_ER_pos=int((s["Metastatic_ER"] == "Positive").sum()),
                     Metastatic_ER_neg=int((s["Metastatic_ER"] == "Negative").sum()),
                     Total=len(s)))
    return pd.DataFrame(rows)


def plot_windows(tab, title, out_stem):
    """Stacked bars (ER+ bottom, ER- top) with n on top. PNG + PDF."""
    fig, ax = plt.subplots(figsize=(10, 5.4))
    x = np.arange(len(tab))
    pos, neg, tot = tab["Metastatic_ER_pos"].to_numpy(), tab["Metastatic_ER_neg"].to_numpy(), tab["Total"].to_numpy()
    ax.bar(x, pos, 0.62, color=COL_POS, edgecolor="#1B4F72", linewidth=0.6, label="Metastatic ER+")
    ax.bar(x, neg, 0.62, bottom=pos, color=COL_NEG, edgecolor="#9C5A14", linewidth=0.6,
           label="Metastatic ER−")
    for i in x:
        ax.text(i, pos[i] / 2, str(pos[i]), ha="center", va="center", color="white",
                fontsize=11, fontweight="bold")
        ax.text(i, pos[i] + neg[i] / 2, str(neg[i]), ha="center", va="center", color="black",
                fontsize=11, fontweight="bold")
        ax.text(i, tot[i] + tot.max() * 0.015, f"n={tot[i]}", ha="center", va="bottom",
                fontsize=11, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels([w.replace("Outside ", "Outside\n") for w in tab["Window"]], fontsize=11)
    ax.set_ylabel("Number of patients", fontsize=11)
    ax.set_ylim(0, tot.max() * 1.2)
    ax.grid(axis="y", color="#E6E6E6", linewidth=0.7)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.legend(frameon=False, loc="upper left", fontsize=11)
    fig.suptitle(title, x=0.01, ha="left", fontsize=15, fontweight="bold")
    fig.text(0.01, 0.01, "Windows are cumulative (±60 days includes ±30 days). "
                         "Lag = draw day − first informative metastatic ER assessment day.",
             fontsize=8, color="#555555")
    fig.tight_layout(rect=(0, 0.03, 1, 0.94))
    fig.savefig(OUT_DIR / f"{out_stem}.png", dpi=300)
    fig.savefig(OUT_DIR / f"{out_stem}.pdf")
    plt.close(fig)


# =============================================================================
# 1. Load clinical data
# =============================================================================
for p in (CLINICAL_XLSX, META_XLSX):
    if not p.exists():
        raise FileNotFoundError(f"Missing input file: {p}")

sheet = find_clinical_sheet(CLINICAL_XLSX)
clin = pd.read_excel(CLINICAL_XLSX, sheet_name=sheet)
clin["patient_id"] = clin["patient_id"].map(normalize_patient_id)
clin["day"] = pd.to_numeric(clin["Days_From_Primary_Diagnosis"], errors="coerce")
clin["ER"] = clin["ER_Status"].map(normalize_er)
print(f"Clinical sheet: {sheet} | rows {len(clin)} | patients {clin['patient_id'].nunique()}")

# ---- primary tumour ER (First_Primary row) ----
prim = (clin.loc[clin["Category"].eq("First_Primary") & clin["ER"].notna()]
        .groupby("patient_id")["ER"].first().rename("Primary_ER"))

# ---- first informative metastatic ER assessment (Met_Diagnosis rows) ----
met_all = clin.loc[clin["Category"].eq("Met_Diagnosis") & clin["day"].notna()].copy()
earliest_met = met_all.groupby("patient_id")["day"].min().rename("Earliest_Met_Diagnosis_day")
met_inf = met_all.loc[met_all["ER"].notna()].sort_values(["patient_id", "day"])
first_day = met_inf.groupby("patient_id")["day"].min()
at_first = met_inf.merge(first_day.rename("first_day"), on="patient_id")
at_first = at_first.loc[at_first["day"].eq(at_first["first_day"])]
n_status = at_first.groupby("patient_id")["ER"].nunique()
met = pd.DataFrame({
    "Metastatic_ER_day": first_day,
    "Metastatic_ER": at_first.groupby("patient_id")["ER"].first(),
    "Metastatic_ER_site": at_first.groupby("patient_id")["Tissue_Site"].first(),
    "n_informative_Met_ER_rows": met_inf.groupby("patient_id").size(),
    "Met_ER_status_changes_later": met_inf.groupby("patient_id")["ER"].nunique().gt(1)
                                   .map({True: "Yes", False: "No"}),
})
conflict = n_status[n_status > 1].index
if len(conflict):
    print(f"WARNING: {len(conflict)} patients have conflicting ER results on their first "
          f"informative day; set to missing and excluded: {list(conflict)}")
    met.loc[conflict, "Metastatic_ER"] = np.nan

# ---- cfDNA draws that have Griffin data ----
meta = pd.read_excel(META_XLSX)
griffin_samples = set(meta["Collaborator_sample_id"].map(normalize_sample).dropna())
draws = clin.loc[clin["Category"].eq("Blood_Draw") & clin["day"].notna() & clin["Barcode"].notna(),
                 ["patient_id", "Barcode", "day"]].copy()
draws["sample"] = draws["Barcode"].map(normalize_sample)
n_all_draws = len(draws)
draws = draws.loc[draws["sample"].isin(griffin_samples)].drop_duplicates("sample")
draws["draw_no"] = pd.to_numeric(draws["sample"].str.extract(r"-NB(\d+)$")[0], errors="coerce")
print(f"Blood draws in clinical sheet: {n_all_draws} | with Griffin data: {len(draws)} | "
      f"patients: {draws['patient_id'].nunique()}")

# =============================================================================
# 2. Patient-level table: one row per patient, lag for EVERY patient
# =============================================================================
audit = (pd.DataFrame({"patient_id": sorted(draws["patient_id"].unique())})
         .merge(prim, on="patient_id", how="left")
         .merge(earliest_met, on="patient_id", how="left")
         .merge(met, on="patient_id", how="left"))
audit["Primary_ER"] = audit["Primary_ER"].fillna("Unknown")
n_before = len(audit)
audit = audit.loc[audit["Metastatic_ER"].notna()].copy()
print(f"Patients with Griffin draw: {n_before} | with informative metastatic ER: {len(audit)}")

# first Griffin draw (earliest day; tie -> lower NB number)
first = (draws.sort_values(["patient_id", "day", "draw_no"]).groupby("patient_id").first()
         [["sample", "day", "draw_no"]]
         .rename(columns={"sample": "First_draw_sample", "day": "First_draw_day", "draw_no": "First_draw_NB"}))
audit = audit.merge(first, on="patient_id", how="left")

# closest Griffin draw to the ER-defining assessment (tie -> earlier day)
d2 = draws.merge(audit[["patient_id", "Metastatic_ER_day"]], on="patient_id")
d2["abs_lag"] = (d2["day"] - d2["Metastatic_ER_day"]).abs()
closest = (d2.sort_values(["patient_id", "abs_lag", "day"]).groupby("patient_id").first()
           [["sample", "day"]]
           .rename(columns={"sample": "Closest_draw_sample", "day": "Closest_draw_day"}))
audit = audit.merge(closest, on="patient_id", how="left")
audit["n_Griffin_draws"] = audit["patient_id"].map(draws.groupby("patient_id").size())

# lags (days): draw - ER assessment; negative = blood BEFORE the ER assessment
audit["Lag_firstdraw_minus_ERdef_days"] = audit["First_draw_day"] - audit["Metastatic_ER_day"]
audit["Lag_closestdraw_minus_ERdef_days"] = audit["Closest_draw_day"] - audit["Metastatic_ER_day"]
audit["Lag_firstdraw_minus_earliestMet_days"] = audit["First_draw_day"] - audit["Earliest_Met_Diagnosis_day"]
audit["Closest_draw_is_first_draw"] = audit["Closest_draw_sample"].eq(audit["First_draw_sample"])

# ER evolution group (primary -> metastatic)
ev = {("Positive", "Positive"): "Primary ER+ → metastatic ER+ (ER retained)",
      ("Positive", "Negative"): "Primary ER+ → metastatic ER− (acquired ER loss)",
      ("Negative", "Negative"): "Primary ER− → metastatic ER− (stable ER-negative)",
      ("Negative", "Positive"): "Primary ER− → metastatic ER+ (gain)"}
audit["ER_evolution_group"] = [ev.get((p, m), "Primary ER unknown")
                               for p, m in zip(audit["Primary_ER"], audit["Metastatic_ER"])]

# =============================================================================
# 3. Cohort counts, window tables and lag summary
# =============================================================================
print(f"\nCohort: {len(audit)} patients | metastatic ER+ {int((audit['Metastatic_ER'] == 'Positive').sum())}, "
      f"ER− {int((audit['Metastatic_ER'] == 'Negative').sum())}")
print(audit["ER_evolution_group"].value_counts().to_string())

win_first = window_table(audit, "Lag_firstdraw_minus_ERdef_days", "First Griffin draw")
win_closest = window_table(audit, "Lag_closestdraw_minus_ERdef_days", "Closest Griffin draw")
print("\nFirst draw windows:\n" + win_first.drop(columns="Window_days").to_string(index=False))
print("\nClosest draw windows:\n" + win_closest.drop(columns="Window_days").to_string(index=False))

lag_rows = []
for rule, col in (("First Griffin draw", "Lag_firstdraw_minus_ERdef_days"),
                  ("Closest Griffin draw", "Lag_closestdraw_minus_ERdef_days")):
    for grp, sub in [("All patients", audit)] + list(audit.groupby("ER_evolution_group")):
        lag = sub[col]
        lag_rows.append({
            "Draw_rule": rule, "Group": grp, "n": len(sub),
            "Median_lag_days": lag.median(), "Q1_lag_days": lag.quantile(0.25),
            "Q3_lag_days": lag.quantile(0.75),
            "Median_abs_lag_days": lag.abs().median(),
            "n_draw_before_ER_assessment": int((lag < 0).sum()),
            "n_within_90d": int((lag.abs() <= 90).sum()),
            "n_outside_365d": int((lag.abs() > 365).sum())})
lag_summary = pd.DataFrame(lag_rows)

evo = pd.crosstab(audit["ER_evolution_group"], audit["Metastatic_ER"]).reset_index()
changed = int((~audit["Closest_draw_is_first_draw"]).sum())
summary = pd.DataFrame([
    ("Patients with Griffin draw", n_before),
    ("Patients with informative metastatic ER (analysis cohort)", len(audit)),
    ("Metastatic ER+", int((audit["Metastatic_ER"] == "Positive").sum())),
    ("Metastatic ER−", int((audit["Metastatic_ER"] == "Negative").sum())),
    ("Patients whose closest draw differs from first draw", changed),
    ("Patients with conflicting first-day ER (excluded)", len(conflict)),
], columns=["Metric", "Value"])

# =============================================================================
# 4. Save: one workbook + figures (PNG and PDF)
# =============================================================================
audit = audit.sort_values(["ER_evolution_group", "patient_id"]).reset_index(drop=True)
with pd.ExcelWriter(OUT_XLSX) as xw:
    summary.to_excel(xw, sheet_name="Summary", index=False)
    pd.concat([win_first, win_closest]).drop(columns="Window_days").to_excel(
        xw, sheet_name="Timing_windows", index=False)
    lag_summary.to_excel(xw, sheet_name="Lag_summary", index=False)
    evo.to_excel(xw, sheet_name="ER_evolution_groups", index=False)
    audit.to_excel(xw, sheet_name="Patient_audit", index=False)

plot_windows(win_first, "Metastatic ER labels were aligned to the first cfDNA draw",
             "AURORA_ER_timing_first_draw_windows")
plot_windows(win_closest, "Metastatic ER labels were aligned to the closest cfDNA draw",
             "AURORA_ER_timing_closest_draw_windows")
print(f"\nSaved in: {OUT_DIR}")

Clinical sheet: AUP_ClinicalData_20260609.NO.DA | rows 6728 | patients 274
Blood draws in clinical sheet: 769 | with Griffin data: 769 | patients: 262
Patients with Griffin draw: 262 | with informative metastatic ER: 244

Cohort: 244 patients | metastatic ER+ 183, ER− 61
Primary ER+ → metastatic ER+ (ER retained)           179
Primary ER− → metastatic ER− (stable ER-negative)     40
Primary ER+ → metastatic ER− (acquired ER loss)       20
Primary ER unknown                                     3
Primary ER− → metastatic ER+ (gain)                    2

First draw windows:
         Draw_rule            Window  Metastatic_ER_pos  Metastatic_ER_neg  Total
First Griffin draw          ±30 days                 58                 24     82
First Griffin draw          ±60 days                 67                 27     94
First Griffin draw          ±90 days                 71                 29    100
First Griffin draw         ±180 days                 80                 36    116
First Griffi